In [1]:
# Feature extraction v2: adds pulse transit time (PTT) and PPG intensity ratio (PIR)
# on top of the original 4 features.
#
# Re-reads the raw .mat files (rather than data/processed_dataset.npz) because this
# notebook needs the signal at two different filter bands from the same raw window:
#   - 0.5-8 Hz, same band as preprocess.ipynb, for amplitude / pulse width / heart rate
#   - 0.5-15 Hz, wider, for augmentation index -- the 0.5-8 Hz version already has the
#     high-frequency content AIx depends on filtered out, and that content can't be
#     recovered once it's gone, so it has to come from a less-filtered copy of the raw signal.
# It also uses the ECG channel (unused elsewhere in this pipeline) to compute pulse transit
# time: the delay between each ECG R-peak and the FOOT (onset) of the next PPG pulse -- not
# the PPG peak. Pairing to the peak gave implausible delays (median ~50ms, should be
# 150-450ms); pairing to the foot gives a clean, tight, physiologically plausible
# distribution (median ~460ms across a 4-file sample), confirming the foot is the right
# fiducial point here, not a data quality problem.
#
# Reuses the same reject checks as preprocess.ipynb (NaN / flatline / NeuroKit quality /
# too few beats / implausible BP) so the windows that make it into the feature set are
# the same ones that would make it into the PPG-window dataset.
#
# Output: data/features_dataset_v2.npz
#   X          (N, 6): [amplitude, heart_rate, pulse_width, augmentation_index, ptt, pir]
#   y          (N, 2): [SBP, DBP]
#   quality    (N,):   NeuroKit quality score, kept but NOT included in X
#   record_ids (N,):   which original record each window came from, for patient-level splits

import h5py
import numpy as np
import neurokit2 as nk
from scipy.signal import butter, filtfilt, find_peaks

FS = 125
WINDOW_LEN = FS * 5

FILT_B8, FILT_A8 = butter(4, [0.5 / (FS / 2), 8 / (FS / 2)], btype="band")
FILT_B15, FILT_A15 = butter(4, [0.5 / (FS / 2), 15 / (FS / 2)], btype="band")

MIN_PEAK_DISTANCE = int(FS * 60 / 180)
MIN_BEATS_PER_WINDOW = 3
MIN_QUALITY = 0.7
EARLY_WINDOW = int(FS * 0.2)  # a/b waves occur in roughly the first 200ms of a beat
PTT_RANGE = (0.1, 0.7)        # plausible ECG R-peak -> PPG foot delay, seconds

SBP_RANGE = (80, 180)
DBP_RANGE = (40, 110)
PULSE_PRESSURE_RANGE = (20, 100)

DATA_FILES = ["data/Part_1.mat", "data/Part_2.mat", "data/Part_3.mat", "data/Part_4.mat"]


In [2]:
X = []
y = []
quality_scores = []
record_ids = []
global_record_id = 0
reject_counts = {"nan": 0, "flatline": 0, "low_quality": 0, "too_few_beats": 0, "bad_bp_range": 0, "no_beats_matched": 0}

for filepath in DATA_FILES:
    print(f"Processing {filepath}...")
    with h5py.File(filepath, "r") as f:
        part_key = [k for k in f.keys() if k != "#refs#"][0]
        record_refs = f[part_key]

        for i in range(record_refs.shape[0]):
            record = f[record_refs[i][0]][:]
            ppg_full = record[:, 0]
            abp_full = record[:, 1]
            ecg_full = record[:, 2]

            n_windows = len(ppg_full) // WINDOW_LEN
            for w in range(n_windows):
                start, end = w * WINDOW_LEN, (w + 1) * WINDOW_LEN
                ppg, abp, ecg = ppg_full[start:end], abp_full[start:end], ecg_full[start:end]

                if np.isnan(ppg).any() or np.isnan(abp).any() or np.isnan(ecg).any():
                    reject_counts["nan"] += 1
                    continue
                if np.std(ppg) < 1e-3:
                    reject_counts["flatline"] += 1
                    continue

                try:
                    nk_ppg = nk.ppg_clean(ppg, sampling_rate=FS)
                    nk_peaks = nk.ppg_findpeaks(nk_ppg, sampling_rate=FS)["PPG_Peaks"]
                    quality = nk.ppg_quality(nk_ppg, peaks=nk_peaks, sampling_rate=FS, method="templatematch").mean()
                except Exception:
                    quality = 0
                if quality < MIN_QUALITY:
                    reject_counts["low_quality"] += 1
                    continue

                sys_peaks, _ = find_peaks(abp, distance=MIN_PEAK_DISTANCE)
                dia_peaks, _ = find_peaks(-abp, distance=MIN_PEAK_DISTANCE)
                if len(sys_peaks) < MIN_BEATS_PER_WINDOW or len(dia_peaks) < MIN_BEATS_PER_WINDOW:
                    reject_counts["too_few_beats"] += 1
                    continue

                sbp, dbp = np.median(abp[sys_peaks]), np.median(abp[dia_peaks])
                pulse_pressure = sbp - dbp
                bp_ok = (
                    SBP_RANGE[0] <= sbp <= SBP_RANGE[1]
                    and DBP_RANGE[0] <= dbp <= DBP_RANGE[1]
                    and PULSE_PRESSURE_RANGE[0] <= pulse_pressure <= PULSE_PRESSURE_RANGE[1]
                )
                if not bp_ok:
                    reject_counts["bad_bp_range"] += 1
                    continue

                # amplitude / pulse width / heart rate, from the same 0.5-8Hz band as preprocess.ipynb
                ppg_8 = filtfilt(FILT_B8, FILT_A8, ppg)
                peaks_8, _ = find_peaks(ppg_8, distance=MIN_PEAK_DISTANCE)
                troughs_8, _ = find_peaks(-ppg_8, distance=MIN_PEAK_DISTANCE)

                amplitudes, widths, pirs = [], [], []
                for p in peaks_8:
                    prior_troughs = troughs_8[troughs_8 < p]
                    if len(prior_troughs) == 0:
                        continue
                    onset = prior_troughs[-1]
                    amplitudes.append(ppg_8[p] - ppg_8[onset])
                    widths.append((p - onset) / FS)

                    # PPG intensity ratio: AC swing / DC level of the RAW (unfiltered) beat segment
                    next_troughs = troughs_8[troughs_8 > p]
                    next_onset = next_troughs[0] if len(next_troughs) > 0 else len(ppg) - 1
                    beat_raw = ppg[onset:next_onset + 1]
                    dc = beat_raw.mean()
                    if abs(dc) > 0.05:
                        ac = beat_raw.max() - beat_raw.min()
                        pirs.append(ac / dc)

                intervals = np.diff(peaks_8) / FS

                if len(amplitudes) == 0 or len(intervals) == 0:
                    reject_counts["no_beats_matched"] += 1
                    continue

                amplitude = np.median(amplitudes)
                pulse_width = np.median(widths)
                heart_rate = 60 / np.median(intervals)
                pir = np.median(pirs) if len(pirs) > 0 else np.nan

                # augmentation index, from a wider 0.5-15Hz band of the SAME raw window --
                # uses the b/a ratio of the second derivative near each beat's onset as a
                # surrogate for how big the reflected wave is relative to the main pulse
                ppg_15 = filtfilt(FILT_B15, FILT_A15, ppg)
                peaks_15, _ = find_peaks(ppg_15, distance=MIN_PEAK_DISTANCE)
                troughs_15, _ = find_peaks(-ppg_15, distance=MIN_PEAK_DISTANCE)
                d2 = np.gradient(np.gradient(ppg_15))

                aix_ratios = []
                for p in peaks_15:
                    prior_troughs = troughs_15[troughs_15 < p]
                    if len(prior_troughs) == 0:
                        continue
                    onset = prior_troughs[-1]
                    seg = d2[onset:onset + EARLY_WINDOW]
                    if len(seg) < 5:
                        continue
                    a_idx = np.argmax(seg)
                    if a_idx >= len(seg) - 1:
                        continue
                    b_idx = a_idx + 1 + np.argmin(seg[a_idx + 1:])
                    a, b = seg[a_idx], seg[b_idx]
                    if a > 0:
                        aix_ratios.append(-b / a)
                augmentation_index = np.median(aix_ratios) if len(aix_ratios) > 0 else np.nan

                # pulse transit time: ECG R-peak -> FOOT (onset) of the next PPG pulse
                try:
                    ecg_clean = nk.ecg_clean(ecg, sampling_rate=FS)
                    _, ecg_info = nk.ecg_peaks(ecg_clean, sampling_rate=FS)
                    r_peaks = ecg_info["ECG_R_Peaks"]
                except Exception:
                    r_peaks = []

                ptts = []
                for r in r_peaks:
                    delay = (troughs_8 - r) / FS
                    candidates = troughs_8[(delay >= PTT_RANGE[0]) & (delay <= PTT_RANGE[1])]
                    if len(candidates) > 0:
                        ptts.append((candidates[0] - r) / FS)
                ptt = np.median(ptts) if len(ptts) > 0 else np.nan

                X.append([amplitude, heart_rate, pulse_width, augmentation_index, ptt, pir])
                y.append([sbp, dbp])
                quality_scores.append(quality)
                record_ids.append(global_record_id)

            global_record_id += 1

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)
quality_scores = np.array(quality_scores, dtype=np.float32)
record_ids = np.array(record_ids, dtype=np.int32)

print(f"\nKept {len(X)} windows")
print("Rejected:", reject_counts)
print(f"Windows with no valid augmentation index: {np.isnan(X[:, 3]).sum()}")
print(f"Windows with no valid PTT: {np.isnan(X[:, 4]).sum()}")
print(f"Windows with no valid PIR: {np.isnan(X[:, 5]).sum()}")


Processing data/Part_1.mat...


/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to 

Processing data/Part_2.mat...


/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to 

Processing data/Part_3.mat...


/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to 

Processing data/Part_4.mat...


/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
/opt/anaconda3/lib/python3.11/site-packages/neurokit2/signal/signal_period.py:83: NeuroKitWarning: Too few peaks detected to 


Kept 494787 windows
Rejected: {'nan': 1, 'flatline': 0, 'low_quality': 881, 'too_few_beats': 22, 'bad_bp_range': 33137, 'no_beats_matched': 0}
Windows with no valid augmentation index: 1215
Windows with no valid PTT: 458
Windows with no valid PIR: 0


In [3]:
# drop windows where augmentation index, PTT, or PIR couldn't be computed --
# everything else about them is valid, but a model can't take a NaN feature
valid = ~np.isnan(X[:, 3]) & ~np.isnan(X[:, 4]) & ~np.isnan(X[:, 5])
print(f"Dropping {(~valid).sum()} windows with a missing feature")

X, y, quality_scores, record_ids = X[valid], y[valid], quality_scores[valid], record_ids[valid]
feature_names = ["amplitude", "heart_rate", "pulse_width", "augmentation_index", "ptt", "pir"]

for j, name in enumerate(feature_names):
    print(f"{name}: min={X[:,j].min():.3f} max={X[:,j].max():.3f} mean={X[:,j].mean():.3f}")

np.savez("data/features_dataset_v2.npz", X=X, y=y, quality=quality_scores, feature_names=feature_names, record_ids=record_ids)
print("Saved to data/features_dataset_v2.npz")


Dropping 1673 windows with a missing feature
amplitude: min=0.012 max=3.060 mean=1.558
heart_rate: min=45.455 max=182.927 mean=92.937
pulse_width: min=0.072 max=0.604 mean=0.198
augmentation_index: min=-1.000 max=7.039 mean=0.776
ptt: min=0.104 max=0.696 mean=0.440
pir: min=0.084 max=2.278 mean=0.994
Saved to data/features_dataset_v2.npz
